# 4주차 실습. 모델이 무엇을 보고 맞히나, 그 설명은 맞나

1–3주차에는 **얼마나 맞히나**(MAE)와 **얼마나 믿나**(σ)를 봤다. 오늘은 **왜 그렇게 예측했나**를 본다.
물성은 logP 로 한다. logP 에는 Wildman–Crippen (1999) **원자별 기여값**이 있어서, 원자 기여 설명이 맞는지 **정답과 비교**할 수 있다.

| 절 | 내용 | 방법 |
|---|---|---|
| a | 기술자 모델 전역 설명 | 선형 계수, permutation importance, SHAP, PDP/ICE |
| b | Morgan 지문 모델 | SHAP 상위 비트 → 부분구조 그림, 비트 충돌 |
| c | GIN 원자 기여 | gradient saliency, gradient × input, Integrated Gradients, GNNExplainer, readout 분해 |
| d | **설명 검증** | 분자마다 Spearman(설명, Crippen 원자 기여), 무작위·원소 규칙 기준선, 학습 안 한 모델 |
| e | 처음부터 해석되는 모델 | 원자 기여 선형 모델(group contribution), 얕은 나무, KAN |
| f | 함정 | 상관된 기술자, SHAP ≠ 인과, seed 바꾸면 설명이 바뀜 |
| g | 팀 과제 | 축 E(해석), B(물리 결합) |

이 폴더의 `datamanager.py`, `models.py` 와 새 모듈 `xai.py` 를 import 해서 쓴다. CPU 약 8분 (기술자 계산 1분 20초, GIN 학습 2회 약 2분, 설명 검증 약 2분).

In [ ]:
PROP = 'logP'     # 원자 기여 정답(Crippen)은 logP 에만 있다. 다른 물성은 c 절까지 돌아가고 d 절은 해석 불가
MAX_N = 3000      # 수업용. GIN 학습 약 1분 (CPU)
SEED = 42
N_EVAL = 200      # d 절에서 검증할 test 분자 수

import time, json, os, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from rdkit import Chem
from rdkit.Chem import Draw, Crippen
from scipy.stats import spearmanr
warnings.filterwarnings('ignore')
torch.set_num_threads(8)
plt.rcParams.update({'font.size': 12, 'axes.grid': False})
from xai import *
T0 = time.time()
R = {}            # 결과 숫자 모음 (마지막 셀에서 저장)

## 0. 데이터와 분할
2주차 `train.py` 와 같은 분할(80:10:10, random, seed 42)을 기술자·지문·GIN 모두에 쓴다. 같은 test 분자로 비교하기 위해서다.

In [ ]:
sets, scaler = prepare_graphs(PROP, max_n=MAX_N, seed=SEED)
smi = {k: [d.smiles for d in v] for k, v in sets.items()}
y = {k: torch.cat([d.y_raw for d in v]).numpy().ravel() for k, v in sets.items()}
R['data'] = dict(prop=PROP, n=sum(len(v) for v in sets.values()), **{f'n_{k}': len(v) for k, v in sets.items()}, seed=SEED)
R['data']

**정답 역할을 할 Crippen 원자 기여.** RDKit `_CalcCrippenContribs` 는 수소를 따로 센다. 수소 몫을 붙어 있는 원자에 더하면 합이 `MolLogP` 와 같다.
Crippen 도 실험값에 맞춘 **모델**이다. 실험 logP 를 얼마나 맞히는지 먼저 본다.

In [ ]:
s0 = 'CC(=O)Nc1ccc(Cl)cc1'
print('원자 기여 :', crippen_atom_contribs(s0).round(2))
print('합 =', crippen_atom_contribs(s0).sum().round(4), '  MolLogP =', round(Crippen.MolLogP(Chem.MolFromSmiles(s0)), 4))
crip_te = np.array([Crippen.MolLogP(Chem.MolFromSmiles(s)) for s in smi['test']])
R['crippen_test_MAE'] = float(np.mean(np.abs(crip_te - y['test'])))
R['mean_baseline_MAE'] = float(np.mean(np.abs(y['train'].mean() - y['test'])))
print(f"Crippen MolLogP 의 test MAE = {R['crippen_test_MAE']:.3f}   (평균값 예측 = {R['mean_baseline_MAE']:.3f})")

## a. 기술자 모델: 어떤 기술자가 중요한가 (전역 설명)
1주차 기술자 217개에는 `MolLogP`(Crippen logP 자체)가 들어 있었다. logP 를 예측하면서 logP 추정값을 넣으면 설명은 "MolLogP 가 중요하다"로 끝난다.
그래서 **Crippen 값으로 만든 기술자 28개**(`MolLogP`, `MolMR`, `SlogP_VSA1–12`, `SMR_VSA1–10`, `BCUT2D_LOGP*`, `BCUT2D_MR*`)를 뺀다.

In [ ]:
t = time.time()
X = {k: descriptor_matrix(v)[0] for k, v in smi.items()}
names = descriptor_matrix(smi['test'][:1])[1]
print(f'기술자 {len(names)}개 (Crippen 계열 {217 - len(names)}개 제외), {time.time() - t:.0f} s')

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
mae = lambda a, b: float(np.mean(np.abs(a - b)))

ridge = make_pipeline(StandardScaler(), Ridge(alpha=10.0)).fit(X['train'], y['train'])
xgb = XGBRegressor(n_estimators=600, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.5,
                   n_jobs=8, random_state=SEED).fit(X['train'], y['train'])
R['MAE'] = {'Ridge_desc': mae(ridge.predict(X['test']), y['test']), 'XGB_desc': mae(xgb.predict(X['test']), y['test'])}
R['MAE']

### a-1. 선형 계수
기술자를 표준화(평균 0, 표준편차 1)한 뒤의 계수는 "기술자가 1 표준편차 커질 때 logP 변화"다. 크기로 순위를 매긴다.

In [ ]:
coef = ridge[-1].coef_
imp = pd.DataFrame({'name': names, 'ridge_abs_coef': np.abs(coef), 'ridge_coef': coef})
imp.sort_values('ridge_abs_coef', ascending=False).head(10)

### a-2. Permutation importance
test 세트에서 기술자 한 열만 섞고 MAE 가 얼마나 나빠지는지 본다. 모델 종류와 상관없이 쓸 수 있다(model-agnostic).

In [ ]:
from sklearn.inspection import permutation_importance
t = time.time()
pi = permutation_importance(xgb, X['test'], y['test'], scoring='neg_mean_absolute_error', n_repeats=5, random_state=SEED, n_jobs=1)
imp['perm'] = pi.importances_mean
print(f'{time.time() - t:.0f} s')
imp.sort_values('perm', ascending=False).head(10)[['name', 'perm']]

### a-3. SHAP
Shapley 값: 기술자들을 하나씩 "넣어 가며" 예측이 얼마나 바뀌는지를, 넣는 **모든 순서**에 대해 평균 낸 몫이다.
분자 하나의 예측 = 평균 예측 + (기술자별 SHAP 값의 합). 나무 모델은 `TreeExplainer` 로 빠르고 정확하게 계산한다 (Lundberg & Lee 2017).

In [ ]:
import shap
ex = shap.TreeExplainer(xgb)
sv = ex.shap_values(X['test'])
print('완전성 확인: 평균 예측 + SHAP 합 = 모델 예측 ?',
      np.allclose(ex.expected_value + sv.sum(1), xgb.predict(X['test']), atol=1e-3))
imp['shap'] = np.abs(sv).mean(0)
shap.summary_plot(sv, X['test'], feature_names=names, max_display=12, show=False)
plt.tight_layout(); plt.show()

**읽는 법.** 점 하나 = test 분자 하나. 가로 = 그 분자에서 그 기술자의 SHAP 값(logP 단위), 색 = 기술자 값(빨강 큼, 파랑 작음).
`TPSA`(극성 표면적)가 크면(빨강) 왼쪽(logP 감소)에 있으면 화학 상식과 맞는 것이다.

### a-4. 세 방법의 순위 비교
같은 모델(XGBoost)이어도 permutation 과 SHAP 은 순위가 다를 수 있다. 선형 계수는 다른 모델(Ridge)의 설명이다.

In [ ]:
imp['xgb_gain'] = xgb.feature_importances_
cols = ['ridge_abs_coef', 'perm', 'shap', 'xgb_gain']
top = {c: imp.sort_values(c, ascending=False).name.head(10).tolist() for c in cols}
display_top = pd.DataFrame(top); display_top.index += 1
rank_rho = pd.DataFrame([[spearmanr(imp[a], imp[b]).correlation for b in cols] for a in cols], index=cols, columns=cols).round(2)
overlap = pd.DataFrame([[len(set(top[a]) & set(top[b])) for b in cols] for a in cols], index=cols, columns=cols)
print('기술자 전체 순위 Spearman'); print(rank_rho)
print('\n상위 10개 겹치는 수'); print(overlap)
display_top

### a-5. PDP / ICE
SHAP 상위 기술자 둘에 대해, 그 값만 바꾸고 나머지는 그대로 둘 때 예측이 어떻게 변하는지 그린다.
얇은 선 = 분자 하나 (ICE), 굵은 선 = 평균 (PDP). 선들이 평행하지 않으면 다른 기술자와 상호작용이 있다는 뜻이다.

In [ ]:
from sklearn.inspection import partial_dependence
top2 = imp.sort_values('shap', ascending=False).index[:2].tolist()
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
pdp_store = {}
for ax, j, tag in zip(axes, top2, 'ab'):
    pd_res = partial_dependence(xgb, X['test'], [j], kind='both', grid_resolution=30)
    grid = pd_res['grid_values'][0]; ice = pd_res['individual'][0]; avg = pd_res['average'][0]
    rng = np.random.default_rng(0); sel = rng.choice(len(ice), 60, replace=False)
    for k in sel:
        ax.plot(grid, ice[k], color='#56B4E9', lw=0.6, alpha=0.5)
    ax.plot(grid, avg, color='#D55E00', lw=2.5, label='PDP (mean)')
    ax.plot([], [], color='#56B4E9', lw=1, label='ICE (one molecule)')
    ax.set_xlabel(names[j]); ax.set_ylabel('predicted logP'); ax.legend(frameon=False, loc='best')
    ax.text(-0.12, 1.04, tag, transform=ax.transAxes, fontsize=14, fontweight='bold')
    pdp_store[names[j]] = dict(grid=grid.tolist(), avg=avg.tolist(), ice=ice[sel].tolist())
plt.tight_layout(); plt.show()

## b. Morgan 지문 모델: 비트를 부분구조로 되돌리기
Morgan 비트 하나 = "어떤 원자를 중심으로 반경 r 안의 부분구조"를 해시한 번호다. SHAP 상위 비트를 골라 **어떤 부분구조인지 그림으로** 되돌린다.

In [ ]:
XF = {k: morgan_matrix(v) for k, v in smi.items()}
xgb_fp = XGBRegressor(n_estimators=600, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.5,
                      n_jobs=8, random_state=SEED).fit(XF['train'], y['train'])
R['MAE']['XGB_morgan'] = mae(xgb_fp.predict(XF['test']), y['test'])
sv_fp = shap.TreeExplainer(xgb_fp).shap_values(XF['test'].astype(float))
bit_imp = np.abs(sv_fp).mean(0)
top_bits = np.argsort(-bit_imp)[:6]
rows = []
for b in top_bits:
    on = XF['test'][:, b] == 1
    frags = bit_fragments(smi['train'], int(b))
    rows.append(dict(bit=int(b), mean_abs_shap=bit_imp[b], shap_when_on=sv_fp[on, b].mean() if on.any() else np.nan,
                     frac_on=on.mean(), n_fragment_kinds=len(frags), top_fragment=list(frags)[0] if frags else ''))
bits = pd.DataFrame(rows).round(3)
print(f"Morgan + XGBoost test MAE = {R['MAE']['XGB_morgan']:.3f}")
bits

In [ ]:
imgs, legends = [], []
for b in top_bits:
    for s in smi['train']:
        m = Chem.MolFromSmiles(s); bi = morgan_bit_info(m)
        if int(b) in bi:
            im = Draw.DrawMorganBit(m, int(b), bi, useSVG=False)
            imgs.append(png_to_array(im) if isinstance(im, bytes) else im); legends.append(f'bit {b}'); break
fig, axes = plt.subplots(1, len(imgs), figsize=(2.4 * len(imgs), 2.6))
for ax, im, lg in zip(axes, imgs, legends):
    ax.imshow(im); ax.set_title(lg, fontsize=11); ax.axis('off')
plt.tight_layout(); plt.show()

**읽는 법.** 파란 원 = 중심 원자, 노란 원 = 방향족 원자, 회색 = 반경 안 원자. `n_fragment_kinds` 가 2 이상이면 서로 다른 부분구조가 **같은 비트**를 켠 것(비트 충돌)이다.
그 비트의 SHAP 값은 여러 부분구조의 효과가 섞인 값이다.

## c. GIN 원자 기여
readout 은 **sum** 으로 한다. logP 처럼 원자 몫을 더해 만드는 물성에 맞고(2주차 0.41 → 0.31), 아래 'readout 분해'가 정확히 성립한다.
`xai.train_gnn` 은 `train.run()` 과 같은 학습 설정(GIN 5층, hidden 128, dropout 0.3, AdamW 1e-3, batch 64)에 모델 객체를 돌려준다.

In [ ]:
model, info = train_gnn(sets, scaler, pooling='sum', seed=SEED, epochs=150, patience_stop=20)
R['gin'] = info; R['MAE']['GIN_sum'] = info['test_MAE']
info

원자 v 의 기여 $a_v$ (특징 147개를 더해 원자 하나에 숫자 하나):

| 방법 | 식 | 부호 | 한 줄 직관 |
|---|---|---|---|
| saliency | $\sum_f \lvert \partial \hat y / \partial x_{vf} \rvert$ | 없음 | 이 원자 특징을 살짝 바꾸면 예측이 얼마나 흔들리나 |
| gradient × input | $\sum_f x_{vf}\, \partial \hat y / \partial x_{vf}$ | 있음 | 1차 테일러 근사의 원자 몫 |
| Integrated Gradients | $\sum_f x_{vf} \int_0^1 \partial \hat y(\alpha x)/\partial x_{vf}\, d\alpha$ | 있음 | 0 에서 x 까지 걸어가며 모은 기울기. 합 = $\hat y(x) - \hat y(0)$ |
| GNNExplainer | 예측을 유지하는 최소 원자 마스크 | 없음 | 원자를 지워도 예측이 같으면 중요하지 않다 |
| readout 분해 | $w \cdot z_v$ | 있음 | sum pool + 선형 head 이면 예측 = 원자 몫의 합 그대로 |

In [ ]:
d = sets['test'][0]
ig, chk = attr_integrated_gradients(model, d, return_check=True)
print(d.smiles)
print('IG 완전성: sum IG =', round(chk['sum_IG'], 3), ' F(x) - F(0) =', round(chk['F_x_minus_F_baseline'], 3))
R['ig_completeness'] = chk
methods = {'saliency':     lambda d: attr_saliency(model, d),
           'grad_x_input': lambda d: attr_grad_input(model, d),
           'IG':           lambda d: attr_integrated_gradients(model, d),
           'GNNExplainer': lambda d: attr_gnnexplainer(model, d, epochs=100),
           'readout':      lambda d: attr_readout(model, d),
           'element_rule': lambda d: attr_element_rule(d.smiles),
           'random':       None}
pd.DataFrame({'atom': [a.GetSymbol() for a in Chem.MolFromSmiles(d.smiles).GetAtoms()],
              'Crippen': crippen_atom_contribs(d.smiles),
              **{k: f(d) for k, f in methods.items() if f is not None and k != 'element_rule'}}).round(3)

## d. 설명 검증: Crippen 원자 기여와 순위가 맞나
분자마다 "원자 기여 순위"가 Crippen 순위와 얼마나 같은지 Spearman ρ 로 잰다 (1 = 같은 순위, 0 = 무관, −1 = 반대).
- **random**: 원자마다 무작위 숫자. 설명이 이것보다 못하면 의미가 없다
- **element_rule**: 모델 없이 "C·S·할로젠 +1, N·O −1". 화학 상식 수준이 어디까지 맞는지
- 부호가 없는 방법(saliency, GNNExplainer)은 **|기여| vs |Crippen|** 열도 같이 본다

시간: 분자 200개 × 방법 7개, 약 2분 (GNNExplainer 가 대부분).

In [ ]:
t = time.time()
ev, store = evaluate_attributions(model, sets['test'], methods, n=N_EVAL, seed=SEED)
summ = summarize(ev)
R['eval_sec'] = round(time.time() - t, 1); R['eval'] = summ.reset_index().to_dict('records')
print(f'{time.time() - t:.0f} s,  분자 {ev.i.nunique()}개 (원자 6개 이상)')
summ.sort_values('signed_mean', ascending=False)

In [ ]:
order = summ.sort_values('signed_mean', ascending=False).index.tolist()
fig, ax = plt.subplots(figsize=(9, 4))
data = [ev[ev.method == m].rho_signed.dropna().values for m in order]
ax.boxplot(data, showfliers=False); ax.set_xticks(range(1, len(order) + 1)); ax.set_xticklabels(order)
for k, v in enumerate(data):
    ax.scatter(np.full(len(v), k + 1) + np.random.default_rng(k).uniform(-0.15, 0.15, len(v)), v, s=5, alpha=0.3, color='#0072B2')
ax.axhline(0, color='k', lw=0.8)
ax.set_ylabel('Spearman rho vs Crippen (per molecule)')
plt.xticks(rotation=20); plt.tight_layout(); plt.show()

**같은 4개 분자를 방법별로 그려 보기.** 원자 6–20개 분자 중 IG 의 ρ 가 가장 높은 것, 중간 둘, 가장 낮은 것을 고른다 (잘 맞는 경우와 틀린 경우 모두).
주황 = logP 를 올리는 쪽, 보라 = 내리는 쪽. 분자마다 최댓값으로 정규화했다.

In [ ]:
cand = ev[(ev.method == 'IG') & ev.n_atoms.between(6, 20)].dropna().sort_values('rho_signed', ascending=False)
pick = [cand.i.iloc[0], cand.i.iloc[len(cand) // 3], cand.i.iloc[2 * len(cand) // 3], cand.i.iloc[-1]]
show = ['Crippen', 'readout', 'IG', 'grad_x_input', 'GNNExplainer']
fig, axes = plt.subplots(len(pick), len(show), figsize=(2.6 * len(show), 2.3 * len(pick)))
examples = []
for r, k in enumerate(pick):
    s = ev[(ev.i == k)].smiles.iloc[0]
    w = {'Crippen': crippen_atom_contribs(s), **{m: store[(k, m)] for m in show[1:]}}
    examples.append(dict(i=int(k), smiles=s, **{m: np.asarray(v).tolist() for m, v in w.items()},
                         rho={m: float(ev[(ev.i == k) & (ev.method == m)].rho_signed.iloc[0]) for m in show[1:]}))
    for c, m in enumerate(show):
        ax = axes[r, c]; ax.imshow(png_to_array(atom_map_png(s, w[m]))); ax.set_xticks([]); ax.set_yticks([])
        rho = '' if m == 'Crippen' else f"  rho={examples[-1]['rho'][m]:.2f}"
        if r == 0 or m != 'Crippen':
            ax.set_title(m + rho, fontsize=10)
plt.tight_layout(); plt.show()

**sanity check: 학습 안 한 모델 (model randomization test).** 가중치가 무작위인 GIN 으로 같은 계산을 한다.
설명 방법이 모델이 배운 것을 반영한다면 ρ 가 크게 떨어져야 한다. 떨어지지 않으면 그 방법은 **입력(원자 종류)만 보고 있는** 것이다 (Adebayo et al. 2018).

In [ ]:
torch.manual_seed(0)
untrained = GraphRegressor(147, 16, conv='GIN', dropout=0.3, pooling='sum').eval()
m_un = {'grad_x_input': lambda d: attr_grad_input(untrained, d), 'IG': lambda d: attr_integrated_gradients(untrained, d),
        'readout': lambda d: attr_readout(untrained, d), 'saliency': lambda d: attr_saliency(untrained, d)}
ev_un, _ = evaluate_attributions(untrained, sets['test'], m_un, n=N_EVAL, seed=SEED)
cmp = pd.DataFrame({'trained': summ.signed_mean, 'untrained': summarize(ev_un).signed_mean}).dropna().round(3)
R['untrained'] = cmp.reset_index().rename(columns={'index': 'method'}).to_dict('records')
cmp

## e. 처음부터 해석되는 모델 (intrinsically interpretable)
사후 설명(post-hoc) 대신 **모델 자체가 설명**인 경우.

1. **원자 기여 선형 모델 (group contribution)**: 분자 벡터 = 원자 특징 147개의 합, 그 위에 Ridge. 원자 v 의 몫 = $x_v \cdot w$ 가 정확한 설명이다.
   화공에서 쓰는 Joback 법, Crippen 법과 같은 형태이고, GIN 을 0층으로 줄인 것과 같다.
2. **얕은 결정나무 (깊이 3)**: 규칙 8개로 끝나서 그림 한 장으로 읽힌다.
3. **KAN (Kolmogorov–Arnold network)**: 연결선마다 1차원 함수(스플라인)를 학습. 가지치기 후 함수 모양을 직접 보거나 식으로 바꿀 수 있다.
   연구실 KAN-DoE 원고(투고 중)는 KAN 의 미분으로 기술자 중요도(average gradient energy)를 계산해 실험 설계에 넣는다. 여기서는 코드 없이 덱에서만 다룬다.

In [ ]:
from sklearn.tree import DecisionTreeRegressor, plot_tree
A = {k: np.stack([d.x.sum(0).numpy() for d in v]) for k, v in sets.items()}           # 원자 특징 합 (분자 x 147)
gc = Ridge(alpha=1.0).fit(A['train'], y['train'])
R['MAE']['GroupContrib_Ridge'] = mae(gc.predict(A['test']), y['test'])
tree = DecisionTreeRegressor(max_depth=3, random_state=SEED).fit(X['train'], y['train'])
R['MAE']['Tree_depth3_desc'] = mae(tree.predict(X['test']), y['test'])
R['MAE']['Crippen_MolLogP'] = R['crippen_test_MAE']

w = gc.coef_
gc_rho = [rank_agreement(d.x.numpy() @ w, crippen_atom_contribs(d.smiles))
          for d in [d for d in sets['test'] if d.x.shape[0] >= 6][:N_EVAL]]
R['group_contrib_rho'] = dict(mean=float(np.nanmean(gc_rho)), sd=float(np.nanstd(gc_rho, ddof=1)))
print(f"group contribution 원자 몫 vs Crippen: rho = {np.nanmean(gc_rho):.3f} ± {np.nanstd(gc_rho, ddof=1):.3f}")
pd.Series(R['MAE']).sort_values().round(3).to_frame('test MAE')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
plot_tree(tree, feature_names=names, filled=False, impurity=False, precision=2, fontsize=9, ax=ax)
plt.show()
from feature import NODE_FEATURE_NAMES
freq = np.concatenate([d.x.numpy() for d in sets['train']]).mean(0)            # 학습 원자 중 그 칸이 1인 비율
gcw = pd.Series(w, index=NODE_FEATURE_NAMES)[freq >= 0.02]                       # 드문 칸(원자 2% 미만)은 계수가 불안정해서 뺀다
gc_top = pd.concat([gcw.nlargest(6), gcw.nsmallest(6)]).round(3)
print('원자 특징 계수 (logP / 원자, 원자 2% 이상에서 켜지는 칸만) 상위 +, 하위 -'); print(gc_top)
print('원소 칸:', gcw[[n for n in gcw.index if n.startswith('element=')]].round(3).to_dict())
R['gc_top'] = {k: float(v) for k, v in gc_top.items()}
R['gc_elements'] = {k: float(v) for k, v in gcw[[n for n in gcw.index if n.startswith('element=')]].items()}

## f. 함정 셋
### f-1. 상관된 기술자는 중요도를 나눠 가진다
SHAP 1위 기술자를 그대로 복사한 열(작은 잡음만 추가)을 하나 더 넣고 다시 학습한다. 모델 성능은 같은데 중요도가 두 열로 갈라진다.

In [ ]:
j = imp.shap.idxmax(); rng = np.random.default_rng(0)
dup = {k: np.c_[v, v[:, j] + 1e-3 * v[:, j].std() * rng.standard_normal(len(v))] for k, v in X.items()}
xgb_dup = XGBRegressor(n_estimators=600, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.5,
                       n_jobs=8, random_state=SEED).fit(dup['train'], y['train'])
sv_dup = np.abs(shap.TreeExplainer(xgb_dup).shap_values(dup['test'])).mean(0)
R['dup'] = dict(feature=names[j], before=float(imp.shap[j]), after_original=float(sv_dup[j]), after_copy=float(sv_dup[-1]),
                MAE_before=R['MAE']['XGB_desc'], MAE_after=mae(xgb_dup.predict(dup['test']), y['test']))
R['dup']

### f-2. SHAP ≠ 인과
SHAP 은 **이 모델이** 그 기술자 값에 얼마나 반응했는지다. 분자에서 그 기술자만 바꾸는 실험이 불가능한 경우가 많고(`TPSA` 를 바꾸면 원자 수도 바뀐다),
상관된 기술자 중 무엇이 원인인지는 데이터로 가를 수 없다. "TPSA 를 줄이면 logP 가 오른다"는 화학 지식이 따로 있을 때만 맞는 말이다.

### f-3. seed 를 바꾸면 설명이 바뀐다
같은 데이터·같은 설정에서 seed 만 바꿔 다시 학습하고, 설명끼리 순위가 얼마나 같은지 본다.

In [ ]:
xgb2 = XGBRegressor(n_estimators=600, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.5,
                    n_jobs=8, random_state=SEED + 1).fit(X['train'], y['train'])
shap2 = np.abs(shap.TreeExplainer(xgb2).shap_values(X['test'])).mean(0)
t10a, t10b = set(np.argsort(-imp.shap.values)[:10]), set(np.argsort(-shap2)[:10])
R['seed_xgb'] = dict(rho_all=float(spearmanr(imp.shap, shap2).correlation), top10_overlap=len(t10a & t10b),
                     MAE_seed2=mae(xgb2.predict(X['test']), y['test']))
print('XGBoost seed 42 vs 43:', R['seed_xgb'])

model2, info2 = train_gnn(sets, scaler, pooling='sum', seed=7, epochs=150, patience_stop=20)
R['gin_seed2'] = info2
pool = [d for d in sets['test'] if d.x.shape[0] >= 6][:N_EVAL]
seed_rows = []
for d in pool:
    ref = crippen_atom_contribs(d.smiles)
    for name, f in [('IG', attr_integrated_gradients), ('readout', attr_readout), ('grad_x_input', attr_grad_input)]:
        a1, a2 = f(model, d), f(model2, d)
        seed_rows.append(dict(method=name, rho_seed=rank_agreement(a1, a2), rho_crippen_seed2=rank_agreement(a2, ref)))
seed_df = pd.DataFrame(seed_rows).groupby('method').agg(['mean', 'std']).round(3)
R['seed_gin'] = {m: dict(rho_seed_mean=float(seed_df.loc[m, ('rho_seed', 'mean')]), rho_seed_sd=float(seed_df.loc[m, ('rho_seed', 'std')]),
                         rho_crippen_seed2=float(seed_df.loc[m, ('rho_crippen_seed2', 'mean')])) for m in seed_df.index}
seed_df

## g. 팀 과제 (5주차 축 선택과 연결)
**축 E (해석)**
1. 팀 물성으로 a 절을 돌려 SHAP 상위 10개를 표로 낸다. 화학적으로 말이 되는 것 / 안 되는 것을 나눈다
2. 팀 물성에 원자·작용기 기여 정답이 있는지 찾는다 (예: 끓는점·임계물성 Joback 기여). 있으면 d 절처럼 Spearman 으로 GIN 설명을 검증한다
3. 설명 방법 하나를 골라 seed 3개로 다시 돌리고 순위 일치도를 보고한다. 일치도가 낮으면 그 설명으로 결론을 내지 않는다

**축 B (물리 결합)**
4. e 절의 group contribution 모델을 팀 물성에 맞춰 본다. GIN 과 MAE 차이가 얼마인지, 그 차이를 메우는 것이 물리 항(예: logS = f(logP, mp))인지 확인한다

**규칙**: 설명 그림을 결과로 쓸 때는 (i) 무작위 기준선, (ii) 학습 안 한 모델, (iii) seed 2개 이상을 같이 보고한다.

In [ ]:
os.makedirs('results_xai', exist_ok=True)
R['runtime_sec'] = round(time.time() - T0, 1)
imp.to_csv('results_xai/desc_importance.csv', index=False)
bits.to_csv('results_xai/morgan_top_bits.csv', index=False)
ev.to_csv('results_xai/rho_per_molecule.csv', index=False)
np.savez_compressed('results_xai/shap_desc.npz', sv=sv, X=X['test'], names=np.array(names))
json.dump(dict(R=R, examples=examples, pdp=pdp_store, top_bits=[int(b) for b in top_bits],
               rank_rho=rank_rho.to_dict(), overlap=overlap.to_dict(), top10=top),
          open('results_xai/xai_summary.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1, default=float)
print(f"전체 {R['runtime_sec']:.0f} s")